# Segmentation-Guided ResNet-50 — Google Colab

This notebook runs `003_classification/segmentation_guided_cv_resnet50/train.py` on a Colab GPU. A single profile switch selects the experiment, dataset layout, and CT input type. Only the selected CT directories, CV metadata, ground-truth masks, and matching U-Net probability maps are extracted from the archive. Training outputs are written directly to Google Drive so they persist after the runtime ends.

Before opening Colab, create the archive from the repository root:

```bash
# Windowed CT profile (dc730a13...)
tar -czf dc730a13-5813-4d87-b15c-3b630deb32b5_segmentation_guided_cv_resnet50.tar.gz \
  000_dataset/_segmentation_dataset_v2/004_classification_cv_5fold_seed42.csv \
  000_dataset/_segmentation_dataset_v2/ct_windowed \
  000_dataset/_segmentation_dataset_v2/mask \
  experiment_results/dc730a13-5813-4d87-b15c-3b630deb32b5/segmentation/unet/inference/probability_npy
sha256sum dc730a13-5813-4d87-b15c-3b630deb32b5_segmentation_guided_cv_resnet50.tar.gz > dc730a13-5813-4d87-b15c-3b630deb32b5_segmentation_guided_cv_resnet50.tar.gz.sha256

# Lung-parenchyma profile (242d4058...)
tar -czf 242d4058-fee2-47cb-b1f2-6608348300f5_segmentation_guided_cv_resnet50.tar.gz \
  000_dataset/_segmentation_dataset_v2/004_classification_cv_5fold_seed42.csv \
  000_dataset/_segmentation_dataset_v2/ct_parenchyma \
  000_dataset/_segmentation_dataset_v2/mask \
  experiment_results/242d4058-fee2-47cb-b1f2-6608348300f5/segmentation/unet/inference/probability_npy
sha256sum 242d4058-fee2-47cb-b1f2-6608348300f5_segmentation_guided_cv_resnet50.tar.gz > 242d4058-fee2-47cb-b1f2-6608348300f5_segmentation_guided_cv_resnet50.tar.gz.sha256

# Windowed CT from the split LIDC/LNDb dataset_v2 layout (0877cfde...)
tar -czf 0877cfde-8744-4aaf-909b-7f906a240117_segmentation_guided_cv_resnet50.tar.gz \
  000_dataset_v2/_segmentation_dataset/004_classification_cv_5fold_seed42.csv \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_windowed \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_windowed \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask \
  experiment_results/0877cfde-8744-4aaf-909b-7f906a240117/segmentation/unet/inference/probability_npy
sha256sum 0877cfde-8744-4aaf-909b-7f906a240117_segmentation_guided_cv_resnet50.tar.gz > 0877cfde-8744-4aaf-909b-7f906a240117_segmentation_guided_cv_resnet50.tar.gz.sha256
```

Upload both files to the Google Drive folder configured in section 3. Make sure the source-code changes have also been pushed to the selected GitHub branch.

> Training runs five folds and may take longer than a single Colab runtime. The script saves a checkpoint after every epoch, but it does not yet provide a CLI for resuming an interrupted CV run. Use a sufficiently long runtime and keep the session open during training.

## 1. Enable and verify the GPU

Select **Runtime > Change runtime type > GPU** before running this cell.

In [ ]:
import shutil
import torch
import os

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Enable it through Runtime > Change runtime type."
    )

disk = shutil.disk_usage("/content")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print(f"Disk free: {disk.free / 2**30:.1f} GiB")
print("CPU cores:", os.cpu_count())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Runtime configuration

Set only `TRAINING_PROFILE` to switch between registered experiments. Each profile binds one UUID, dataset layout, CT source, metadata column, and JSON configuration. Reduce `BATCH_SIZE` if CUDA runs out of memory.

In [ ]:
from pathlib import Path

# Source code
REPOSITORY_URL = "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Change only this value to switch the complete experiment profile.
TRAINING_PROFILE = "windowed_v2_0877"

TRAINING_PROFILES = {
    "windowed": {
        "experiment_id": "dc730a13-5813-4d87-b15c-3b630deb32b5",
        "ct_input_type": "windowed",
        "ct_path_column": "ct_windowed_path",
        "dataset_container": "000_dataset",
        "dataset_root": "000_dataset/_segmentation_dataset_v2",
        "metadata_path": (
            "000_dataset/_segmentation_dataset_v2/"
            "004_classification_cv_5fold_seed42.csv"
        ),
        "ct_members": [
            "000_dataset/_segmentation_dataset_v2/ct_windowed"
        ],
        "mask_members": [
            "000_dataset/_segmentation_dataset_v2/mask"
        ],
        "config_filename": "segmentation_guided_cv_resnet50_windowed.json",
    },
    "parenchyma": {
        "experiment_id": "242d4058-fee2-47cb-b1f2-6608348300f5",
        "ct_input_type": "parenchyma",
        "ct_path_column": "ct_parenchyma_path",
        "dataset_container": "000_dataset",
        "dataset_root": "000_dataset/_segmentation_dataset_v2",
        "metadata_path": (
            "000_dataset/_segmentation_dataset_v2/"
            "004_classification_cv_5fold_seed42.csv"
        ),
        "ct_members": [
            "000_dataset/_segmentation_dataset_v2/ct_parenchyma"
        ],
        "mask_members": [
            "000_dataset/_segmentation_dataset_v2/mask"
        ],
        "config_filename": "segmentation_guided_cv_resnet50_parenchyma.json",
    },
    "windowed_v2_0877": {
        "experiment_id": "0877cfde-8744-4aaf-909b-7f906a240117",
        "ct_input_type": "windowed",
        "ct_path_column": "ct_windowed_path",
        "dataset_container": "000_dataset_v2",
        "dataset_root": "000_dataset_v2/_segmentation_dataset",
        "metadata_path": (
            "000_dataset_v2/_segmentation_dataset/"
            "004_classification_cv_5fold_seed42.csv"
        ),
        "ct_members": [
            (
                "000_dataset_v2/_lidc/"
                "007_segmentation_dataset_npy/ct_windowed"
            ),
            (
                "000_dataset_v2/_lndb/"
                "007_segmentation_dataset_npy/ct_windowed"
            ),
        ],
        "mask_members": [
            (
                "000_dataset_v2/_lidc/"
                "007_segmentation_dataset_npy/mask"
            ),
            (
                "000_dataset_v2/_lndb/"
                "007_segmentation_dataset_npy/mask"
            ),
        ],
        "config_filename": (
            "segmentation_guided_cv_resnet50_windowed_v2_0877.json"
        ),
    },
}
if TRAINING_PROFILE not in TRAINING_PROFILES:
    raise ValueError(
        f"TRAINING_PROFILE must be one of {sorted(TRAINING_PROFILES)}"
    )
profile = TRAINING_PROFILES[TRAINING_PROFILE]
EXPERIMENT_ID = profile["experiment_id"]
CT_INPUT_TYPE = profile["ct_input_type"]
CT_PATH_COLUMN = profile["ct_path_column"]
DATASET_CONTAINER = profile["dataset_container"]
DATASET_ROOT_RELATIVE = Path(profile["dataset_root"])
METADATA_PATH_RELATIVE = Path(profile["metadata_path"])
CT_MEMBERS = tuple(profile["ct_members"])
MASK_MEMBERS = tuple(profile["mask_members"])
PROFILE_CONFIG_FILENAME = profile["config_filename"]

# Archive uploaded to Google Drive
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/mask-guided-lung-nodule-xai")
ARCHIVE_NAME = (
    f"{EXPERIMENT_ID}_segmentation_guided_cv_resnet50.tar.gz"
)

DRIVE_DATASET_ARCHIVE = DRIVE_PROJECT_DIR / ARCHIVE_NAME

DRIVE_DATASET_CHECKSUM = (
    DRIVE_PROJECT_DIR / f"{ARCHIVE_NAME}.sha256"
)

# Extraction location on Colab local storage
LOCAL_DATA_ROOT = Path("/content/classification_data")

# Persistent checkpoints and metrics
DRIVE_EXPERIMENT_ROOT = (
    DRIVE_PROJECT_DIR / "experiment_results" / EXPERIMENT_ID
)
DRIVE_GUIDED_OUTPUT_DIR = (
    DRIVE_EXPERIMENT_ROOT / "classification/guided_resnet50"
)

# Colab training overrides
BATCH_SIZE = 64       # reduce to 16, 8, or 4 if an OOM occurs
NUM_WORKERS = 0  # safest on Colab; increase only after a stable smoke test
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 5

# Set to True only when re-extracting data in the same runtime
FORCE_REEXTRACT = False

print(f"Profile        : {TRAINING_PROFILE}")
print(f"Experiment ID  : {EXPERIMENT_ID}")
print(f"CT input type  : {CT_INPUT_TYPE}")
print(f"CT path column : {CT_PATH_COLUMN}")
print(f"Dataset root   : {DATASET_ROOT_RELATIVE}")
print(f"Archive        : {DRIVE_DATASET_ARCHIVE}")
print(f"Output         : {DRIVE_GUIDED_OUTPUT_DIR}")

## 4. Clone or update the repository

The GitHub branch must already contain `003_classification/segmentation_guided_cv_resnet50`.

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

required_script = (
    PROJECT_ROOT
    / "003_classification/segmentation_guided_cv_resnet50/train.py"
)
profile_config_path = (
    PROJECT_ROOT / "003_classification/configs" / PROFILE_CONFIG_FILENAME
)
if not required_script.is_file():
    raise FileNotFoundError(
        f"Training script not found: {required_script}. "
        "Push the local changes to the GitHub branch first."
    )
if not profile_config_path.is_file():
    raise FileNotFoundError(
        f"Profile configuration not found: {profile_config_path}. "
        "Push the profile JSON files to the selected branch first."
    )
print(f"Repository ready: {PROJECT_ROOT}")

## 5. Install dependencies

The Torch and Torchvision versions bundled with Colab are retained for CUDA compatibility. Zennit is required when running LRP from `test.py`.

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import sys

required_versions = {
    "albumentations": "2.0.8",
    "zennit": "0.5.1",
}
packages = []
for package, required_version in required_versions.items():
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        packages.append(f"{package}=={required_version}")

if packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )

required_modules = (
    "torch", "torchvision", "albumentations", "cv2", "numpy",
    "pandas", "sklearn", "matplotlib", "tqdm",
)
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Required dependencies are unavailable: {missing}")
print("All dependencies are ready.")

## 6. Validate and extract the selected experiment archive

The archive can use either supported dataset layout. This cell extracts only the members registered by the selected profile:

- `004_classification_cv_5fold_seed42.csv`
- one or more `ct_windowed/` or `ct_parenchyma/` directories
- one or more ground-truth nodule-mask directories
- probability maps from `experiment_results/EXPERIMENT_ID/segmentation/unet`

The archive is read directly from Drive to avoid creating another archive copy on local storage.

In [ ]:
import hashlib

if not DRIVE_DATASET_ARCHIVE.is_file():
    raise FileNotFoundError(f"Archive not found: {DRIVE_DATASET_ARCHIVE}")

print(f"Archive size: {DRIVE_DATASET_ARCHIVE.stat().st_size / 2**30:.2f} GiB")

# The checksum is optional but recommended for large uploads.
if DRIVE_DATASET_CHECKSUM.is_file():
    expected_hash = DRIVE_DATASET_CHECKSUM.read_text().split()[0].strip().lower()
    digest = hashlib.sha256()
    with DRIVE_DATASET_ARCHIVE.open("rb") as archive_file:
        for chunk in iter(lambda: archive_file.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    actual_hash = digest.hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError("Archive checksum mismatch; the upload may be corrupted.")
    print("SHA-256 checksum is valid.")
else:
    print("Checksum not found; SHA-256 validation skipped.")

probability_member = (
    f"experiment_results/{EXPERIMENT_ID}/"
    "segmentation/unet/inference/probability_npy"
)
required_members = (
    str(METADATA_PATH_RELATIVE),
    *CT_MEMBERS,
    *MASK_MEMBERS,
    probability_member,
)
extract_marker = (
    LOCAL_DATA_ROOT
    / f".guided_{EXPERIMENT_ID}_{TRAINING_PROFILE}_extracted"
)

if FORCE_REEXTRACT or not extract_marker.is_file():
    print("Extracting the required training data...", flush=True)
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "tar", "--checkpoint=5000",
            "--checkpoint-action=echo=Extract checkpoint %u",
            "-xzf", str(DRIVE_DATASET_ARCHIVE),
            "-C", str(LOCAL_DATA_ROOT), *required_members,
        ],
        check=True,
    )
    extract_marker.touch()
    print("Extraction complete.")
else:
    print("Local data has already been extracted; reusing it.")

## 7. Link local data and Google Drive outputs

Symlinks provide the dataset layout expected by `train.py` without duplicating data. Guided-classification outputs are written directly to `experiment_results/<UUID>/classification/guided_resnet50` in Google Drive.

In [ ]:
import os

local_dataset_container = LOCAL_DATA_ROOT / DATASET_CONTAINER
local_experiment_dir = (
    LOCAL_DATA_ROOT / "experiment_results" / EXPERIMENT_ID
)
local_segmentation_component = local_experiment_dir / "segmentation"

for required_dir in (local_dataset_container, local_segmentation_component):
    if not required_dir.is_dir():
        raise FileNotFoundError(f"Extracted directory not found: {required_dir}")

drive_classification_component = DRIVE_EXPERIMENT_ROOT / "classification"
drive_classification_component.mkdir(parents=True, exist_ok=True)
project_experiment_root = PROJECT_ROOT / "experiment_results" / EXPERIMENT_ID
project_experiment_root.mkdir(parents=True, exist_ok=True)
project_dataset_link = PROJECT_ROOT / DATASET_CONTAINER
links = {
    project_dataset_link: local_dataset_container,
    project_experiment_root / "segmentation": local_segmentation_component,
    project_experiment_root / "classification": drive_classification_component,
}

for link, target in links.items():
    if link.is_symlink():
        if link.resolve() != target.resolve():
            raise RuntimeError(f"Symlink points to a different target: {link}")
    elif link.exists():
        raise FileExistsError(
            f"Path already exists and is not a symlink: {link}. Use a clean Colab clone."
        )
    else:
        os.symlink(target, link, target_is_directory=True)
    print(f"{link} -> {target}")

## 8. Dataset and model preflight

This cell checks every metadata path, verifies that all probability maps are present, and runs one sample through the model before long-running training starts.

In [ ]:
import importlib
import json
import sys
import pandas as pd

project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string not in sys.path:
    sys.path.insert(0, project_root_string)
importlib.invalidate_caches()

with profile_config_path.open("r", encoding="utf-8") as file:
    profile_config = json.load(file)
profile_data = profile_config["data"]
expected_profile_values = {
    "experiment_id": profile_config["experiment"]["id"],
    "ct_input_type": profile_data["ct_input_type"],
    "ct_path_column": profile_data["ct_path_column"],
    "dataset_root": profile_data["dataset_root"],
    "metadata_path": profile_data["metadata_path"],
}
selected_profile_values = {
    "experiment_id": EXPERIMENT_ID,
    "ct_input_type": CT_INPUT_TYPE,
    "ct_path_column": CT_PATH_COLUMN,
    "dataset_root": str(DATASET_ROOT_RELATIVE),
    "metadata_path": str(METADATA_PATH_RELATIVE),
}
if expected_profile_values != selected_profile_values:
    raise RuntimeError(
        "The selected notebook profile does not match its JSON: "
        f"{selected_profile_values} != {expected_profile_values}"
    )

dataset_root = LOCAL_DATA_ROOT / DATASET_ROOT_RELATIVE
metadata_path = LOCAL_DATA_ROOT / METADATA_PATH_RELATIVE
probability_root = (
    local_experiment_dir / "segmentation/unet/inference/probability_npy"
)

metadata = pd.read_csv(metadata_path)
missing_ct = [
    path
    for path in metadata[CT_PATH_COLUMN]
    if not (dataset_root / str(path)).is_file()
]
missing_probability = [
    name for name in metadata["filename"]
    if not (probability_root / Path(str(name)).name).is_file()
]
missing_mask = [
    path for path in metadata["mask_path"]
    if not (dataset_root / str(path)).is_file()
]
if missing_ct or missing_probability or missing_mask:
    raise FileNotFoundError(
        f"Missing CT={len(missing_ct)}, mask={len(missing_mask)}, "
        f"probability={len(missing_probability)}"
    )

dataset_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.dataset"
)
transform_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.transforms"
)
model_module = importlib.import_module(
    "003_classification.segmentation_guided_cv_resnet50.model"
)

validation_dataset = (
    dataset_module.ProbabilityGuidedClassificationDataset(
        root_dir=dataset_root,
        metadata_path=metadata_path,
        split="val",
        cv_fold=0,
        probability_root=probability_root,
        ct_path_column=CT_PATH_COLUMN,
        transform=transform_module.build_val_transform(
            int(profile_data["input_height"]),
            int(profile_data["input_width"]),
            tuple(profile_data["normalization_mean"]),
            tuple(profile_data["normalization_std"]),
            int(profile_config["training"]["transform_seed"]),
        ),
    )
)
sample, target = validation_dataset[0]
smoke_model = model_module.SegmentationGuidedResNet50(
    num_classes=2, weights=None
).to("cuda").eval()
with torch.no_grad():
    smoke_output = smoke_model(sample.unsqueeze(0).to("cuda"))
del smoke_model
torch.cuda.empty_cache()

print(f"Training profile    : {TRAINING_PROFILE}")
print(f"Experiment ID       : {EXPERIMENT_ID}")
print(f"CT input type       : {CT_INPUT_TYPE}")
print(f"CT path column      : {CT_PATH_COLUMN}")
print(f"Dataset root        : {dataset_root}")
print(f"Metadata path       : {metadata_path}")
print(f"Metadata rows       : {len(metadata):,}")
print(f"Validation fold 0   : {len(validation_dataset):,}")
print(f"Input sample        : {tuple(sample.shape)}")
print(f"Target              : {int(target)}")
print(f"Model output        : {tuple(smoke_output.shape)}")
print(f"Probability maps    : {len(list(probability_root.glob('*.npy'))):,}")
print(f"Ground-truth masks  : {len(metadata) - len(missing_mask):,}")
print("Preflight completed successfully.")

## 9. Run five-fold training

This cell creates a separate runtime JSON from the selected profile and passes it to `train.py --config`. The repository configuration is never modified. `train.py` copies the effective runtime JSON to the Google Drive output. Each UUID has one `classification/guided_resnet50` directory, and the script refuses to overwrite an existing directory. If an OOM occurs before training starts successfully, reduce `BATCH_SIZE` and restart the runtime to fully clear VRAM.

In [ ]:
import json

with profile_config_path.open("r", encoding="utf-8") as file:
    training_config = json.load(file)

# Build an effective runtime config without editing the Git clone.
training_config["experiment"]["id"] = EXPERIMENT_ID
training_config["data"]["dataset_root"] = str(DATASET_ROOT_RELATIVE)
training_config["data"]["metadata_path"] = str(METADATA_PATH_RELATIVE)
training_config["data"]["ct_input_type"] = CT_INPUT_TYPE
training_config["data"]["ct_path_column"] = CT_PATH_COLUMN
training_config["data"]["probability_root"] = (
    f"experiment_results/{EXPERIMENT_ID}/"
    "segmentation/unet/inference/probability_npy"
)
training_config["training"]["batch_size"] = BATCH_SIZE
training_config["training"]["num_epochs"] = NUM_EPOCHS
training_config["training"]["device"] = "cuda"
training_config["dataloader"]["num_workers"] = NUM_WORKERS
training_config["dataloader"]["persistent_workers"] = (
    NUM_WORKERS > 0
)
training_config["early_stopping"]["patience"] = (
    EARLY_STOPPING_PATIENCE
)
runtime_config_dir = LOCAL_DATA_ROOT / "runtime_configs"
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config_path = (
    runtime_config_dir
    / f"{EXPERIMENT_ID}_segmentation_guided_cv_resnet50.json"
)
with runtime_config_path.open("w", encoding="utf-8") as file:
    json.dump(training_config, file, indent=4)
    file.write("\n")

project_output_dir = (
    PROJECT_ROOT / "experiment_results" / EXPERIMENT_ID
    / "classification/guided_resnet50"
)
if project_output_dir.resolve() != DRIVE_GUIDED_OUTPUT_DIR.resolve():
    raise RuntimeError(
        f"Unexpected output path: {project_output_dir}"
    )

training_command = [
    sys.executable,
    "-m",
    "003_classification.segmentation_guided_cv_resnet50.train",
    "--config",
    str(runtime_config_path),
]
print(f"Runtime config : {runtime_config_path}")
print(f"Profile        : {TRAINING_PROFILE}")
print(f"CT input type  : {CT_INPUT_TYPE}")
print(f"CT path column : {CT_PATH_COLUMN}")
print(f"Dataset root   : {DATASET_ROOT_RELATIVE}")
print(f"Output run     : {project_output_dir}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Epoch/fold     : {NUM_EPOCHS}")
print(f"Patience       : {EARLY_STOPPING_PATIENCE}")
subprocess.run(training_command, cwd=PROJECT_ROOT, check=True)

## 10. Inspect training outputs

In [ ]:
from IPython.display import display

latest_run = DRIVE_GUIDED_OUTPUT_DIR
if not latest_run.is_dir():
    raise FileNotFoundError(f"No training output found at {latest_run}")
print(f"Guided classification: {latest_run}")

summary_path = latest_run / "cv_summary.csv"
if summary_path.is_file():
    display(pd.read_csv(summary_path))
else:
    completed_folds = sorted(
        path.name for path in latest_run.glob("fold_*")
        if (path / "best_model.pth").is_file()
    )
    print(f"Training is incomplete. Folds with saved models: {completed_folds}")

## 11. Run testing, Grad-CAM, and LRP

This cell verifies that ground-truth masks are available and then evaluates `experiment_results/EXPERIMENT_ID/classification/guided_resnet50`. By default, the complete holdout set is evaluated. LRP is computationally expensive; set `MAX_TEST_SAMPLES` to a small number such as `8` for a smoke test. `TEST_NUM_WORKERS=0` improves stability on Colab/Python 3.13. Visualizations are saved as one PNG per study in `test/visualization/`, with a separate section for each nodule.

In [ ]:
import subprocess
import sys

# None selects the configured completed run. Set a Path to select another run.
TEST_RUN_DIR = None
MAX_TEST_SAMPLES = None  # example: 8 for a smoke test
TEST_BATCH_SIZE = 1  # conservative for five-model XAI inference
TEST_NUM_WORKERS = 0  # safest setting for Colab/Python 3.13
TEST_DPI = 120  # suitable for potentially tall study canvases

# Study visualizations require ground-truth masks. An older archive may
# have been extracted before masks were added to the training-data list.
holdout_metadata = pd.read_csv(metadata_path)
holdout_metadata = holdout_metadata.loc[
    holdout_metadata["cv_role"].astype(str).str.lower().eq("holdout_test")
]
missing_masks = [
    dataset_root / str(path)
    for path in holdout_metadata["mask_path"]
    if not (dataset_root / str(path)).is_file()
]
if missing_masks:
    print(
        f"Extracting ground-truth masks ({len(missing_masks):,} missing)...",
        flush=True,
    )
    subprocess.run(
        [
            "tar", "-xzf", str(DRIVE_DATASET_ARCHIVE),
            "-C", str(LOCAL_DATA_ROOT), *MASK_MEMBERS,
        ],
        check=True,
    )
    missing_masks = [
        dataset_root / str(path)
        for path in holdout_metadata["mask_path"]
        if not (dataset_root / str(path)).is_file()
    ]
if missing_masks:
    raise FileNotFoundError(
        f"Ground-truth masks are still missing: {len(missing_masks):,}"
    )
print(f"Ground-truth masks ready: {len(holdout_metadata):,}")

required_folds = tuple(range(5))

if TEST_RUN_DIR is None:
    test_run_dir = DRIVE_GUIDED_OUTPUT_DIR
else:
    test_run_dir = Path(TEST_RUN_DIR).expanduser().resolve()

missing_checkpoints = [
    test_run_dir / f"fold_{fold}/best_model.pth"
    for fold in required_folds
    if not (test_run_dir / f"fold_{fold}/best_model.pth").is_file()
]
if missing_checkpoints:
    raise FileNotFoundError(
        "Fold checkpoints are incomplete:\n"
        + "\n".join(str(path) for path in missing_checkpoints)
    )

command = [
    sys.executable, "-m",
    "003_classification.segmentation_guided_cv_resnet50.test",
    str(test_run_dir),
    "--batch-size", str(TEST_BATCH_SIZE),
    "--num-workers", str(TEST_NUM_WORKERS),
    "--device", "cuda",
    "--dpi", str(TEST_DPI),
]
if MAX_TEST_SAMPLES is not None:
    command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

print(f"Testing run : {test_run_dir}")
print("Running ensemble inference, Grad-CAM, and LRP...", flush=True)

subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print(f"Test output         : {test_run_dir / 'test'}")
print(f"Study visualization: {test_run_dir / 'test/visualization'}")